# 🧪 [Day 31] Cypher 집계·인덱스·실행계획(PROFILE) 실전 워크북

- **과정 구분**: 지식그래프 엔지니어링 실전 마스터
- **데이터셋**: [DART-Trace] 상장사 대주주 지분 데이터 & [ART:READY] 미대입시 전형 데이터
- **핵심 미션**: SUM/COUNT/AVG 통계 집계, B-Tree 인덱스 생성, EXPLAIN/PROFILE을 통한 NodeIndexSeek 성능 최적화를 직접 실습한다.

## 1. 환경 설정 및 드라이버 연결

In [ ]:
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv()
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USERNAME", os.getenv("NEO4J_USER", "neo4j"))
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
print("✅ Neo4j 연결 성공:", NEO4J_URI)

## 2. [DART-Trace] 기업별 대주주 총 지분율 및 평균 집계 (SUM, AVG, COUNT)

In [ ]:
dart_agg_cypher = """
MATCH (s:Shareholder)-[r:HOLDS_ECONOMIC_STAKE]->(c:Company)
RETURN 
    c.name AS company,
    count(s) AS total_shareholders,
    round(sum(r.stake_ratio), 2) AS total_stake_ratio,
    round(avg(r.stake_ratio), 2) AS avg_stake_ratio,
    max(r.stake_ratio) AS max_stake_ratio
ORDER BY total_stake_ratio DESC;
"""

with driver.session() as session:
    records = list(session.run(dart_agg_cypher))
    for r in records:
        print(f"[{r['company']}] 대주주: {r['total_shareholders']}명 | 총 지분: {r['total_stake_ratio']}% (평균: {r['avg_stake_ratio']}%)")

## 3. [ART:READY] 대학별 개설 전형 수 및 평균 실기비중 랭킹

In [ ]:
art_agg_cypher = """
MATCH (u:University)-[:OFFERS_TRACK]->(t:AdmissionTrack)-[r:REQUIRES_PRACTICAL]->(p:PracticalType)
RETURN 
    u.name AS university,
    u.campus AS campus,
    count(DISTINCT t) AS track_count,
    round(avg(r.ratio), 1) AS avg_practical_ratio,
    max(r.ratio) AS max_practical_ratio
ORDER BY avg_practical_ratio DESC, track_count DESC;
"""

with driver.session() as session:
    records = list(session.run(art_agg_cypher))
    for r in records:
        print(f"[{r['university']} ({r['campus']})] 전형: {r['track_count']}개 | 평균 실기: {r['avg_practical_ratio']}% (최고: {r['max_practical_ratio']}%)")

## 4. [성능 튜닝] B-Tree 인덱스 생성 및 PROFILE 실행 계획 비교

In [ ]:
create_index_cypher = """
CREATE CONSTRAINT constraint_company_corp_code IF NOT EXISTS
FOR (c:Company) REQUIRE c.corp_code IS UNIQUE;
"""

profile_cypher = """
PROFILE
MATCH (c:Company {corp_code: '00126380'})
RETURN c.name, c.market_type;
"""

with driver.session() as session:
    session.run(create_index_cypher)
    print("✅ corp_code 고유 인덱스 제약조건 생성 완료")
    res = session.run(profile_cypher)
    print("✅ PROFILE 쿼리 실행 완료 (NodeIndexSeek 적용)")